# Assignment 5

## Load Data
Load the data from the database created in the previous assignment

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np

In [2]:
load_dotenv()
dcm_user_password = os.getenv('dcm_user_password')

db_config = {
    "user": "dcm_user",
    "password": dcm_user_password, 
    "host": "localhost",
    "port": "5432",
    "dbname": "dcm_db"
}

In [3]:
connection_str = f"postgresql://{db_config['user']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['dbname']}"

In [4]:
engine = create_engine(connection_str)

In [5]:
query = """
SELECT 
    m."Title",
    m."Year",
    STRING_AGG(g."Name", ', ') AS "Genre",
    m."Metascore",
    m."imdbRating",
    y."Is_Recession",
    m."AdjustedBoxOffice",
    m."RottenTomatoes"
FROM public.movies m
LEFT JOIN public.year_recession_info y ON m."Year" = y."Year"
LEFT JOIN public.movie_genres mg ON m."MovieID" = mg."MovieID"
LEFT JOIN public.genres g ON mg."GenreID" = g."GenreID"
GROUP BY 
    m."MovieID", 
    m."Title", 
    m."Year", 
    m."Metascore", 
    m."imdbRating", 
    y."Is_Recession", 
    m."AdjustedBoxOffice", 
    m."RottenTomatoes"
ORDER BY m."MovieID";
"""

try:
    df = pd.read_sql(query, engine)
    display(df.head())
except Exception as e:
    print("Error connecting to database:", e)

,Title,Year,Genre,Metascore,imdbRating,Is_Recession,AdjustedBoxOffice,RottenTomatoes
0,The Shawshank Redemption,1994,Drama,82,9.3,False,62662497.0,89
1,The Dark Knight,2008,"Crime, Drama, Action",85,9.1,True,802459799.0,94
2,Inception,2010,"Action, Adventure, Sci-Fi",74,8.8,False,433190616.0,87
3,Fight Club,1999,"Crime, Drama, Thriller",67,8.8,False,71772007.0,81
4,Interstellar,2014,"Drama, Sci-Fi, Adventure",74,8.7,False,277197045.0,73


## Feature-Engineering and Data-Transformation

### Inflation Adjustment (done in assignment 3)
In the 3rd assignment, a new feature `AdjustedBoxOffice` was engineered by merging the movie dataset with inflation data, joined by release year. The original BoxOffice revenue was multiplied by the ratio of the 2025 CPI to the release-year CPI.

Financial data spanning decades is incomparable because of currency losing its values. Without this transformation, recent movies would appear more successful than older classics solely due to inflation.

### Unstructured Data Extraction (done in assignment 3)

The raw Ratings column contained a list of dictionaries (e.g., [{'Source': 'Rotten Tomatoes', 'Value': '89%'}]). A custom parsing function was applied to iterate through these lists, identify the "`Rotten Tomatoes`" source, strip the percentage sign, and convert the value to an integer.

Only the Rotten Tomatoes column was separated because we already had separated columns for the other 2 ratings (imdbRating and Metascore).

This was done because ML algorithms cannot interpret nested dictionary structures or string based percentages.

### Binary Standardization (done in assignment 3)

The `Is_Recession` feature contained mixed data types (0/1, "True"/"False", "TRUE"/"FALSE", "YES"/"NO"). These were standardized into a single boolean format.

Inconsistent labeling in binary classification features leads to distinct categories being treated as separate classes (e.g., "YES" vs "True"), leading to a loss of power of the variable.

### Multi-Hot Encoding (Genres)

The Genre attribute contained multi-valued strings (eg. "Action, Adventure, Sci-Fi"). We applied Multi-Hot Encoding to split these strings by the comma delimiter. This generated a distinct binary column for every unique genre (e.g., Genre_Action, Genre_Sci-Fi), where a value of 1 indicates the presence of that genre.

A single movie often belongs to multiple genres. Standard One-Hot encoding assumes mutually exclusive categories. Multi-Hot encoding accurately captures the overlapping nature of movie genres.

In [6]:
genre_dummies = df['Genre'].str.get_dummies(sep=', ')
genre_dummies = genre_dummies.add_prefix('Genre_')
df = pd.concat([df, genre_dummies], axis=1)
df.drop(columns=['Genre'], inplace=True)

### Feature Scaling & Normalization

We applied two different scaling techniques depending on the data distribution:

#### Min-Max Normalization
Used for bounded numerical features

- Metascore (0-100)
- imdbRating (0-10)
- RottenTomatoes (0-100)
- Year (1936-2025)

Without scaling, some algorithms would be biased toward features with larger raw numbers, ignoring smaller-scale but highly predictive features like ratings.

#### Logarithmic Transformation
- AdjustedBoxOffice (0-hundreds of millions)

Financial data is typically right-skewed (a few blockbusters earn significantly more than the average). Applying a log transformation normalizes the distribution, reducing the impact of outliers.

In [7]:
scaler = MinMaxScaler()
cols_to_scale = ['Metascore', 'RottenTomatoes', 'imdbRating', 'Year']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

df['AdjustedBoxOffice'] = np.log1p(df['AdjustedBoxOffice'])

### Displaying the final form of the data

In [8]:
df.head()

,Title,Year,Metascore,imdbRating,Is_Recession,AdjustedBoxOffice,RottenTomatoes,Genre_Action,Genre_Adventure,Genre_Animation,...,Genre_Horror,Genre_Music,Genre_Musical,Genre_Mystery,Genre_Romance,Genre_Sci-Fi,Genre_Sport,Genre_Thriller,Genre_War,Genre_Western
0,The Shawshank Redemption,0.586667,0.746479,1.000000,False,17.953274,0.877778,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,The Dark Knight,0.773333,0.788732,0.963636,True,20.503192,0.933333,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Inception,0.800000,0.633803,0.909091,False,19.886688,0.855556,1,1,0,...,0,0,0,0,0,1,0,0,0,0
3,Fight Club,0.653333,0.535211,0.909091,False,18.089005,0.788889,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,Interstellar,0.853333,0.633803,0.890909,False,19.440239,0.700000,0,1,0,...,0,0,0,0,0,1,0,0,0,0
